In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import random
import pickle

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

set_seed(SEED)

device = torch.device("cuda")

In [3]:
class MusicDataset(Dataset):
    def __init__(self):
        with open("pakiet/train.pkl", "rb") as file:
            train_raw = pickle.load(file)
        self.xx = []
        self.yy = []
        for x, y in train_raw:
            self.xx.append(torch.tensor(x, dtype=torch.float32))
            self.yy.append(torch.tensor(y, dtype=torch.long).unsqueeze(0))

    def __len__(self):
        return len(self.xx)

    def __getitem__(self, idx):
        return self.xx[idx], self.yy[idx]

In [4]:
pad_value = 0
def pad_collate(batch):
    xx, yy = zip(*batch)
    yy = torch.stack(yy)
    x_lens = [x.shape[0] for x in xx]
    y_lens = [1] * len(yy)

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=pad_value)

    return xx_pad, yy, x_lens, y_lens

In [5]:
full_trainset = MusicDataset()

train_ratio = 0.8

targets = full_trainset.yy
indices = list(range(len(full_trainset)))

train_indices, val_indices = train_test_split(
    indices,
    train_size=train_ratio,
    stratify=targets,
    random_state=SEED
)

trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

# num_classes = len(trainset.dataset.classes)

# print(f"Number of classes: {num_classes}")
print(f"Trainset size: {len(trainset)}")
print(f"Validation set size: {len(valset)}")

Trainset size: 2351
Validation set size: 588


In [6]:
batch_size = 128
num_workers = 0
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
    collate_fn=pad_collate,
    )
valloader = torch.utils.data.DataLoader(
    valset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    collate_fn=pad_collate,
    )

In [7]:
class LSTMClassifier(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, out_size, dropout_prob=0.3):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True, 
            dropout=dropout_prob if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size * 2, out_size)

    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers * 2, batch_size, self.hidden_size)
        state = torch.zeros(self.num_layers * 2, batch_size, self.hidden_size)
        return hidden, state

    def forward(self, x, hidden):
        packed_outputs, (h_n, c_n) = self.lstm(x, hidden)
        
        forward_hidden = h_n[-2]  
        backward_hidden = h_n[-1] 
        
        last_out = torch.cat((forward_hidden, backward_hidden), dim=1) 
        
        out = self.dropout(last_out)
        out = self.fc(out)
        
        return out

In [8]:
net = LSTMClassifier(1,200,3,5)

In [9]:
epochs = 40
optimizer = torch.optim.AdamW(net.parameters(), lr=0.001)

# max_lr is the peak learning rate it will hit
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=0.01, 
    steps_per_epoch=len(trainloader), 
    epochs=epochs # Must match your total epochs!
)
loss_fun = nn.CrossEntropyLoss()
net.train()
net.to(device)

# Training loop
for epoch in range(epochs):
    for x, targets, x_len, target_len in trainloader:
        x = x.to(device).unsqueeze(2)
        targets = targets.to(device).squeeze(-1)
        hidden, state = net.init_hidden(x.size(0))
        hidden, state = hidden.to(device), state.to(device) 
        
        x_packed = pack_padded_sequence(x, x_len, batch_first=True, enforce_sorted=False)
        preds = net(x_packed, (hidden, state))
        
        optimizer.zero_grad()
        loss = loss_fun(preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
    if epoch % 1 == 0:
        print(f"Epoch: {epoch}, loss: {loss.item():.3}")

Epoch: 0, loss: 1.27
Epoch: 1, loss: 1.26
Epoch: 2, loss: 1.14
Epoch: 3, loss: 0.999
Epoch: 4, loss: 0.949
Epoch: 5, loss: 0.952
Epoch: 6, loss: 1.14
Epoch: 7, loss: 1.01
Epoch: 8, loss: 0.929
Epoch: 9, loss: 1.04
Epoch: 10, loss: 1.04
Epoch: 11, loss: 0.996
Epoch: 12, loss: 1.02
Epoch: 13, loss: 1.1
Epoch: 14, loss: 0.949
Epoch: 15, loss: 0.838
Epoch: 16, loss: 0.902
Epoch: 17, loss: 0.979
Epoch: 18, loss: 0.828
Epoch: 19, loss: 0.732
Epoch: 20, loss: 0.881
Epoch: 21, loss: 0.758
Epoch: 22, loss: 0.573
Epoch: 23, loss: 0.706
Epoch: 24, loss: 0.585
Epoch: 25, loss: 0.701
Epoch: 26, loss: 0.607
Epoch: 27, loss: 0.689
Epoch: 28, loss: 0.495
Epoch: 29, loss: 0.587
Epoch: 30, loss: 0.45
Epoch: 31, loss: 0.399
Epoch: 32, loss: 0.319
Epoch: 33, loss: 0.386
Epoch: 34, loss: 0.289
Epoch: 35, loss: 0.391
Epoch: 36, loss: 0.385
Epoch: 37, loss: 0.288
Epoch: 38, loss: 0.335
Epoch: 39, loss: 0.343


In [ ]:
# # loading
# PATH = "net1.pth"
#
# net.load_state_dict(torch.load(PATH, map_location=device))
# net.to(device)
# net.eval()

LSTMClassifier(
  (lstm): LSTM(1, 200, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=400, out_features=5, bias=True)
)

In [ ]:
# saving
# PATH = "net1.pth"
#
# torch.save(net.state_dict(), PATH)
# print(f"Model weights saved to {PATH}")

Model weights saved to net1.pth


In [12]:
def evaluate_accuracy(model, dataloader, device):
    model.eval()  # Set network to evaluation mode
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():  
        for x, targets, x_len, target_len in dataloader:
            x = x.to(device).unsqueeze(2)
            targets = targets.to(device).squeeze(-1).long()  # Match shape [batch_size]
            
            hidden, state = model.init_hidden(x.size(0))
            hidden, state = hidden.to(device), state.to(device)
            
            x_packed = pack_padded_sequence(x, x_len, batch_first=True, enforce_sorted=False)
            preds = model(x_packed, (hidden, state))
            
            pred_classes = torch.argmax(preds, dim=1)
            
            correct_predictions += (pred_classes == targets).sum().item()
            total_samples += targets.size(0)
            
    accuracy = correct_predictions / total_samples
    return accuracy

In [13]:
evaluate_accuracy(net, trainloader, device)

0.9075520833333334

In [14]:

evaluate_accuracy(net, valloader, device)

0.7227891156462585